In [ ]:
# Load required libraries
library(dplyr)
library(zoo)
library(readr)

# Read the NTL data
ntl_data <- read_csv("22_ntl_2012_2024_monthly.csv", 
                     col_types = cols())

# Convert the 'date' column to a yearmon object
ntl_data <- ntl_data %>%
  mutate(date = as.yearmon(date, "%Y-%m"))

# Filter for odd months (months: 1, 3, 5, 7, 9, 11)
ntl_odd <- ntl_data %>%
  filter(as.numeric(format(date, "%m")) %in% c(1, 3, 5, 7, 9, 11))

# In case there are multiple observations per neighborhood and month,
# calculate the weighted night-time light mean for each neighborhood and date.
ntl_odd_summary <- ntl_odd %>%
  group_by(name, date) %>%
  summarise(
    total_ntl = sum(n_non_na_pixels * ntl_mean, na.rm = TRUE),
    total_pixels = sum(n_non_na_pixels, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(weighted_ntl_mean = total_ntl / total_pixels) %>%
  arrange(name, date) %>%
  group_by(name) %>%
  # Calculate the percentage change compared to the previous odd month for each neighborhood
  mutate(
    prev_weighted_ntl_mean = lag(weighted_ntl_mean),
    pct_change = 100 * (weighted_ntl_mean - prev_weighted_ntl_mean) / prev_weighted_ntl_mean
  ) %>%
  ungroup()

# Save the processed data to a new CSV file
write_csv(ntl_odd_summary, "22_ntl_percentage_change.csv")

# Print the resulting dataset
print(ntl_odd_summary)


# A tibble: 1,716 × 7
   name    date  total_ntl total_pixels weighted_ntl_mean prev_weighted_ntl_mean
   <chr>   <yea>     <dbl>        <dbl>             <dbl>                  <dbl>
 1 Arbutu… 1月 …     111.            37              2.99                  NA   
 2 Arbutu… 3月 …     146.            37              3.96                   2.99
 3 Arbutu… 5月 …     119.            37              3.23                   3.96
 4 Arbutu… 7月 …     115.            37              3.10                   3.23
 5 Arbutu… 9月 …     121.            37              3.27                   3.10
 6 Arbutu… 11月…      96.5           37              2.61                   3.27
 7 Arbutu… 1月 …      89.5           37              2.42                   2.61
 8 Arbutu… 3月 …     138.            37              3.72                   2.42
 9 Arbutu… 5月 …     131.            37              3.53                   3.72
10 Arbutu… 7月 …     131.            37              3.54                   3.53
# ℹ 1,706 more r